<a href="https://colab.research.google.com/github/harini361/Gan-one-credit-/blob/main/DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files

uploaded = files.upload()

Saving archive (1).zip to archive (1) (1).zip


In [4]:
import zipfile
import os

zip_path = next(iter(uploaded.keys()))
extract_path = "/content/WELFake_Data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted files:", os.listdir(extract_path))

Extracted files: ['WELFake_Dataset.csv']


In [10]:

# FAKE NEWS DETECTION USING SIMPLE RNN AND LSTM

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

# 1. Load Dataset
df = pd.read_csv("/content/WELFake_Data/WELFake_Dataset.csv")

print("Dataset Shape:", df.shape)
print("Missing Values:\n", df.isnull().sum())

# 2. Handle Missing Values
df = df.dropna(subset=["label"]).copy()
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["content"] = df["title"] + " " + df["text"]
df = df[df["label"].isin([0, 1])].copy()

# 3. Prepare Input and Output
X = df["content"].astype(str)
y = df["label"].astype(int)

# 4. Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# 5. Tokenization
MAX_WORDS = 10000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# 6. Padding Numerical Sequences
X_train_pad = pad_sequences(
    X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)
X_test_pad = pad_sequences(
    X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)

print("Training Padded Shape:", X_train_pad.shape)
print("Testing Padded Shape:", X_test_pad.shape)

# 7. Simple RNN Model
rnn_model = Sequential([
    Embedding(MAX_WORDS, 64, mask_zero=True),
    SimpleRNN(32),
    Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining Simple RNN Model")
rnn_model.fit(
    X_train_pad, y_train,
    epochs=3, batch_size=64,
    validation_split=0.2
)

rnn_prob = rnn_model.predict(X_test_pad, verbose=0).ravel()
rnn_pred = (rnn_prob >= 0.5).astype(int)

print("Simple RNN Accuracy:",
      accuracy_score(y_test, rnn_pred))

# 8. LSTM Model
lstm_model = Sequential([
    Embedding(MAX_WORDS, 64, mask_zero=True),
    LSTM(32),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining LSTM Model")
lstm_model.fit(
    X_train_pad, y_train,
    epochs=3, batch_size=64,
    validation_split=0.2
)

lstm_prob = lstm_model.predict(X_test_pad, verbose=0).ravel()
lstm_pred = (lstm_prob >= 0.5).astype(int)

print("LSTM Accuracy:",
      accuracy_score(y_test, lstm_pred))

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_model(y_true, y_pred, model_name):
    print("\n", model_name, "Evaluation")
    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("F1 Score :", f1_score(y_true, y_pred, zero_division=0))

# Call the evaluation function for both models
evaluate_model(y_test, rnn_pred, "Simple RNN")
evaluate_model(y_test, lstm_pred, "LSTM")

# 9. Compare Results
print("\nModel Comparison")
print("Simple RNN Accuracy:",
      accuracy_score(y_test, rnn_pred))
print("LSTM Accuracy:",
      accuracy_score(y_test, lstm_pred))

Dataset Shape: (72134, 4)
Missing Values:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64
Training Samples: 57707
Testing Samples: 14427
Training Padded Shape: (57707, 200)
Testing Padded Shape: (14427, 200)

Training Simple RNN Model
Epoch 1/3
722/722 ━━━━━━━━━━━━━━━━━━━━ 71s 94ms/step - accuracy: 0.7884 - loss: 0.4929 - val_accuracy: 0.7453 - val_loss: 0.5373
Epoch 2/3
722/722 ━━━━━━━━━━━━━━━━━━━━ 68s 94ms/step - accuracy: 0.8564 - loss: 0.3642 - val_accuracy: 0.8645 - val_loss: 0.3652
Epoch 3/3
722/722 ━━━━━━━━━━━━━━━━━━━━ 67s 93ms/step - accuracy: 0.8275 - loss: 0.3813 - val_accuracy: 0.6517 - val_loss: 0.6382
Simple RNN Accuracy: 0.6471199833645248

Training LSTM Model
Epoch 1/3
722/722 ━━━━━━━━━━━━━━━━━━━━ 125s 168ms/step - accuracy: 0.8634 - loss: 0.3745 - val_accuracy: 0.8492 - val_loss: 0.4084
Epoch 2/3
722/722 ━━━━━━━━━━━━━━━━━━━━ 117s 163ms/step - accuracy: 0.8848 - loss: 0.3201 - val_accuracy: 0.9122 - val_loss: 0.2724
Epoch 3/3
722/722